# Первая модель: KNN

Авторы BelOAI, CC BY 4.0. Python 3, стандартная библиотека. Загрузите `gliders.csv` рядом с ноутбуком. Набор синтетический; высокая точность здесь не предсказывает результат на реальных планерах.

Последовательность: train для модели и масштаба → validation для выбора → test один раз для финальной оценки.

## Уроки по этим темам

Материал этого ноутбука можно изучать по отдельным урокам:

- [Расстояния и голосование](../../ml/nearest-neighbors/distance-voting/)
- [Масштабирование, выбор и итоговая проверка](../../ml/nearest-neighbors/selection-evaluation/)


In [ ]:
import csv
from pathlib import Path

# В Colab/Jupyter загрузите gliders.csv рядом с ноутбуком.
data_path = Path("gliders.csv")
if not data_path.is_file():
    raise FileNotFoundError("Загрузите gliders.csv в рабочую папку ноутбука")
with data_path.open(encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))
for row in rows:
    row["span_cm"] = float(row["span_cm"])
    row["mass_g"] = float(row["mass_g"])
train = [row for row in rows if row["split"] == "train"]
validation = [row for row in rows if row["split"] == "validation"]
test = [row for row in rows if row["split"] == "test"]
assert (len(train), len(validation), len(test)) == (12, 6, 6)
assert len({row["id"] for row in rows}) == len(rows)
print("train / validation / test:", len(train), len(validation), len(test))


In [ ]:
def accuracy(expected, predicted):
    if len(expected) == 0 or len(expected) != len(predicted):
        raise ValueError("Нужны непустые списки одинаковой длины")
    correct = 0
    for index in range(len(expected)):
        if expected[index] == predicted[index]:
            correct += 1
    return correct / len(expected)


Частоты считаем словарём только по train. Обходим названия классов в алфавитном порядке и заменяем победителя только при строго большей частоте. Поэтому при равенстве остаётся первая метка по алфавиту.

In [ ]:
counts = {}
for row in train:
    label = row["flight"]
    counts[label] = counts.get(label, 0) + 1
baseline = None
for label in sorted(counts):
    if baseline is None or counts[label] > counts[baseline]:
        baseline = label
expected = [row["flight"] for row in validation]
baseline_validation = accuracy(expected, [baseline] * len(validation))
assert baseline_validation == 0.5
print("Baseline:", baseline, "validation accuracy:", baseline_validation)


Функции ниже выполняют отдельные шаги. `features` сохраняет порядок «размах, масса»; `distance` суммирует квадраты разностей и извлекает корень. `fit_scale` получает минимум и диапазон каждого столбца, `transform` применяет сохранённые числа. В `predict` сортируем тройки «расстояние, id, метка», затем считаем голоса. Такой разбор использует обычные циклы и не скрывает порядок разрешения равенств.

In [ ]:
from math import sqrt

def features(row):
    # Только признаки, доступные до предсказания.
    return [row["span_cm"], row["mass_g"]]

def distance(x, z):
    if len(x) != len(z):
        raise ValueError("Число координат должно совпадать")
    squared = 0
    for j in range(len(x)):
        squared += (x[j] - z[j]) ** 2
    return sqrt(squared)

def fit_scale(vectors):
    if len(vectors) == 0 or len(vectors[0]) == 0:
        raise ValueError("Для масштаба нужны объекты с признаками")
    dimensions = len(vectors[0])
    for point in vectors:
        if len(point) != dimensions:
            raise ValueError("Число признаков должно совпадать")
    minima = []
    ranges = []
    for j in range(dimensions):
        values = [point[j] for point in vectors]
        left = min(values)
        width = max(values) - left
        minima.append(left)
        ranges.append(width if width != 0 else 1)
    return minima, ranges

def transform(x, scale):
    minima, ranges = scale
    if len(x) != len(minima) or len(x) != len(ranges):
        raise ValueError("Размер объекта и масштаба должен совпадать")
    return [(x[j] - minima[j]) / ranges[j] for j in range(len(x))]

def predict(query, examples, k):
    if type(k) != int or not 1 <= k <= len(examples):
        raise ValueError("k должен быть целым от 1 до размера train")
    if len(query) == 0:
        raise ValueError("Нужен хотя бы один признак")
    ranked = []
    for row in examples:
        ranked.append((distance(query, row["x"]), row["id"], row["y"]))
    ranked.sort()
    votes = {}
    for value, identifier, label in ranked[:k]:
        votes[label] = votes.get(label, 0) + 1
    winner = None
    for label in sorted(votes):
        if winner is None or votes[label] > votes[winner]:
            winner = label
    return winner

def examples(part, scale=None):
    return [{"id": row["id"], "x": transform(features(row), scale) if scale is not None else features(row), "y": row["flight"]} for row in part]


## Ручной пример

Для q=(15, 10) сравним первые четыре строки train. Эта иллюстрация не заменяет полный train в эксперименте.

In [ ]:
small = examples(train[:4])
print([(row["id"], round(distance([15, 10], row["x"]) ** 2), row["y"]) for row in small])
assert predict([15, 10], small, 1) == "короткий"
assert predict([15, 10], small, 3) == "короткий"

## Выбираем вариант на validation

Сначала перебираем масштабированные данные, затем исходные; внутри — k=1,3,5. Сохраняем новый вариант только при строгом улучшении. Поэтому при равенстве оценок заранее предпочитаем масштабирование и меньшее k. Тест в выборе не участвует.

In [ ]:
# Параметры подготовки оцениваются только на train.
scale = fit_scale([features(row) for row in train])
assert scale == ([12.0, 8.0], [15.0, 30.0])
results = []
best = None
for scaled in (True, False):
    preparation = scale if scaled else None
    fitted = examples(train, preparation)
    queries = examples(validation, preparation)
    for k in (1, 3, 5):
        predicted = [predict(row["x"], fitted, k) for row in queries]
        score = accuracy([row["y"] for row in queries], predicted)
        candidate = {"scaled": scaled, "k": k, "validation_accuracy": score}
        results.append(candidate)
        print(candidate)
        if best is None or score > best["validation_accuracy"]:
            best = candidate
# При равенстве: масштабированный вариант, затем меньшее k.
assert best == {"scaled": True, "k": 1, "validation_accuracy": 1.0}
print("Выбрано без test:", best)


## Финальная оценка

Не изменяйте выбранные параметры после просмотра test.

In [ ]:
# Финальная проверка фиксированного решения; после неё параметры не меняем.
preparation = scale if best["scaled"] else None
fitted = examples(train, preparation)
queries = examples(test, preparation)
predicted = [predict(row["x"], fitted, best["k"]) for row in queries]
test_accuracy = accuracy([row["y"] for row in queries], predicted)
baseline_test = accuracy([row["y"] for row in queries], [baseline] * len(queries))
assert (test_accuracy, baseline_test) == (1.0, 0.5)
print("KNN test:", test_accuracy, "baseline test:", baseline_test)


## Самостоятельные задачи

1. Соседи упорядочены по возрастанию расстояния, их метки A, B, B, A, B. Найдите ответы при k=3 и k=5.
2. Переведите массу в килограммы и сравните масштабированные ответы на данных примера. Проверьте, что диапазон массы по train ненулевой, прежде чем сокращать его в формуле min-max.
3. Объясните, почему проверка k=1 на тех же объектах train обычно даёт 100%, и придумайте исключение.

## Разобранные решения

1. Оба ответа B.
2. При ненулевом обучающем диапазоне массы координата сохраняется: (0.001x−0.001min)/(0.001max−0.001min)=(x−min)/(max−min). Поэтому сохраняются расстояния и ответы. При max=min деление не определено и требуется отдельное правило обработки постоянного признака; сокращать нулевой знаменатель нельзя.
3. Сам объект находится на нулевом расстоянии; это запоминание. Дубликаты с разными метками могут дать неверный ответ из-за правила ID.

In [ ]:
# Решение задачи об единицах: проверяем устойчивость масштабирования.
kg_vectors = [[row["span_cm"], row["mass_g"] / 1000] for row in train]
kg_scale = fit_scale(kg_vectors)
kg_fitted = []
for row in train:
    point = [row["span_cm"], row["mass_g"] / 1000]
    kg_fitted.append({"id": row["id"], "x": transform(point, kg_scale), "y": row["flight"]})
original_fitted = examples(train, scale)
original_queries = examples(validation, scale)
original_answers = [predict(row["x"], original_fitted, 1) for row in original_queries]
kg_answers = []
for row in validation:
    point = [row["span_cm"], row["mass_g"] / 1000]
    kg_answers.append(predict(transform(point, kg_scale), kg_fitted, 1))
assert original_answers == kg_answers
assert distance([1, 2], [4, 6]) == 5
# Постоянный признак: делитель 1; значение вне диапазона допустимо.
assert transform([15, 7], fit_scale([[10, 7], [20, 7]])) == [0.5, 0.0]
assert transform([40], fit_scale([[10], [30]])) == [1.5]
# Одинаковые расстояния: меньший ID; равные голоса: алфавитная метка.
tied = [{"id": "b", "x": [1], "y": "B"}, {"id": "a", "x": [-1], "y": "A"}]
assert predict([0], tied, 1) == "A"
assert predict([0], tied, 2) == "A"
for invalid in (0, 3, 1.5):
    try:
        predict([0], tied, invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("Некорректное k должно отклоняться")


## Итог

KNN улучшил baseline на этой синтетической таблице. Сохраните выводы и помните: шесть проверочных объектов — слишком мало для общих заключений. Продолжение маршрута — качество, утечки и другие модели.